In [ ]:
pip install transformers torch soundfile

In [ ]:
from huggingface_hub import login

login(token="your_hf_token_here")

In [ ]:
import json
import torch
import soundfile as sf
from transformers import VitsModel, AutoTokenizer
from huggingface_hub import login

login(token="your_hf_token_here")

def generate_mms_audio(json_filepath):
    # 1. Load the Meta MMS Urdu TTS Model and Tokenizer
    # MMS TTS is built on the VITS architecture
    print("Loading model and tokenizer...")
    model_id = "facebook/mms-tts-urd-script_arabic"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = VitsModel.from_pretrained(model_id)

    # 2. Load the Dataset
    print(f"Reading {json_filepath}...")
    with open(json_filepath, "r", encoding="utf-8") as file:
        dataset = json.load(file)

    # The JSON structure we created has a "sentences" array
    sentences = dataset.get("sentences", [])

    if not sentences:
        print("Error: Could not find the 'sentences' array in the JSON.")
        return

    # 3. Generate Audio for each sentence
    for item in sentences:
        sentence_id = item["id"]
        text = item["sentence"]

        print(f"Processing Sentence {sentence_id}/{len(sentences)}...")

        # Tokenize the Urdu text
        inputs = tokenizer(text, return_tensors="pt")

        # Generate the waveform (no gradient calculation needed for inference)
        with torch.no_grad():
            output = model(**inputs).waveform

        # 4. Extract and Save the Audio
        # Squeeze the tensor to a 1D array and move to CPU/numpy
        audio_array = output.squeeze().cpu().numpy()

        # VITS models typically output at 16000 Hz or 22050 Hz.
        # Always pull the exact sample rate from the model config to be safe.
        sample_rate = model.config.sampling_rate

        output_filename = f"mms_urdu_sentence_{sentence_id:02d}.wav"
        sf.write(output_filename, audio_array, sample_rate)

    print("\nSuccess! All audio files have been generated.")

# Run the pipeline
generate_mms_audio("urdu_balanced_set.json")

In [ ]:
import os
import shutil
import glob

# 1. Define folder and zip names
folder_name = "mms_urdu_audio"
zip_filename = "mms_urdu_dataset"

# Create the directory if it doesn't exist
os.makedirs(folder_name, exist_ok=True)

# 2. Find and move all generated .wav files into the folder
wav_files = glob.glob("mms_urdu_sentence_*.wav")

if not wav_files:
    print("No .wav files found! Make sure you ran the generation script first.")
else:
    for file in wav_files:
        # Move file into the folder
        shutil.move(file, os.path.join(folder_name, file))

    print(f"Moved {len(wav_files)} audio files into the '{folder_name}/' directory.")

    # 3. Create the ZIP archive
    # shutil.make_archive(base_name, format, root_dir)
    shutil.make_archive(zip_filename, 'zip', folder_name)
    print(f"Successfully compressed into: {zip_filename}.zip")

    # 4. Trigger the download (Specifically for Google Colab)
    try:
        from google.colab import files
        print("Starting download...")
        files.download(f"{zip_filename}.zip")
    except ImportError:
        print(f"Notice: Not running in Google Colab. The file '{zip_filename}.zip' is ready in your current working directory.")


In [ ]:
pip install transformers torch jiwer librosa

In [ ]:
import os
import json
import zipfile
import re
import torch
import jiwer
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration

def normalize_urdu_text(text):
    """
    Strips punctuation and normalizes spaces.
    Crucial for accurate WER/CER calculation in Perso-Arabic scripts.
    """
    # Remove common punctuation marks (including Urdu specific ones like ، and ۔)
    text = re.sub(r'[۔،؛؟,.;?!]', '', text)
    # Remove zero-width non-joiners and zero-width joiners
    text = text.replace('\u200c', '').replace('\u200d', '')
    # Normalize multiple spaces into a single space
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def run_evaluation_pipeline(zip_filepath, json_filepath):
    # 1. Unzip the audio files
    extract_dir = "extracted_audio_for_eval"
    print(f"Unzipping {zip_filepath}...")
    with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    # 2. Load the Ground Truth JSON
    print("Loading ground truth dataset...")
    with open(json_filepath, "r", encoding="utf-8") as f:
        dataset = json.load(f)

    # Create a dictionary mapping ID to the original sentence for easy lookup
    ground_truth_dict = {item["id"]: item["sentence"] for item in dataset["sentences"]}

    # 3. Load Whisper ASR Model (UPDATED TO MEDIUM)
    print("Loading Whisper Medium ASR model for transcription...")
    model_id = "openai/whisper-medium"
    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperForConditionalGeneration.from_pretrained(model_id)

    # Move model to GPU if available in Colab
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    # 4. Process and Transcribe
    references = []
    hypotheses = []

    # Locate all extracted wav files
    wav_files = []
    for root, dirs, files in os.walk(extract_dir):
        for file in files:
            if file.endswith(".wav"):
                wav_files.append(os.path.join(root, file))

    wav_files.sort() # Ensure they are processed in order

    print("\nStarting Transcription and Evaluation...\n")
    for audio_path in wav_files:
        try:
            file_id = int(re.search(r'\d+', os.path.basename(audio_path)).group())
        except AttributeError:
            print(f"Skipping {audio_path}: Could not find ID in filename.")
            continue

        if file_id not in ground_truth_dict:
            print(f"Skipping ID {file_id}: Not found in JSON.")
            continue

        original_text = ground_truth_dict[file_id]

        # Load and resample audio to 16kHz
        audio_array, sr = librosa.load(audio_path, sr=16000)

        # Transcribe
        inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt").input_features.to(device)
        with torch.no_grad():
            forced_decoder_ids = processor.get_decoder_prompt_ids(language="urdu", task="transcribe")
            predicted_ids = model.generate(inputs, forced_decoder_ids=forced_decoder_ids)

        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        # Normalize both texts
        ref_clean = normalize_urdu_text(original_text)
        hyp_clean = normalize_urdu_text(transcription)

        references.append(ref_clean)
        hypotheses.append(hyp_clean)

        print(f"File: {os.path.basename(audio_path)}")
        print(f"  Ref: {ref_clean}")
        print(f"  Hyp: {hyp_clean}")
        print("-" * 50)

    # 5. Calculate Final Metrics
    if not references:
        print("No valid files were processed. Check your filenames and JSON IDs.")
        return

    wer = jiwer.wer(references, hypotheses)
    cer = jiwer.cer(references, hypotheses)

    print("\n" + "="*40)
    print("FINAL EVALUATION METRICS (Whisper Medium)")
    print("="*40)
    print(f"Word Error Rate (WER):      {wer:.4f} ( {wer*100:.2f}% )")
    print(f"Character Error Rate (CER): {cer:.4f} ( {cer*100:.2f}% )")
    print("="*40)

# RUN THE FUNCTION
run_evaluation_pipeline("mms_urdu_dataset.zip", "urdu_balanced_set.json")